If not installed, run:
pip install -U langchain langchain-google-genai chromadb
pip install -U langchain langchain-community langchain-google-genai chromadb

In [1]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

## Indexing

In [2]:
# Load transcripts into a list raw_docs
loader = DirectoryLoader(
    path=r"C:\Users\sacha\jupyter\RAG project\Transcripts",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

raw_docs = loader.load()

print(f"Loaded {len(raw_docs)} documents")
print(raw_docs[0].page_content[:300])
print(raw_docs[0].metadata)

Loaded 10 documents
This is a database class so a natural question is what is a database and why would we even want to use on this video is all about answering those two questions. An initial issue is the word database actually gets used for two different things that we should differentiate one. The database is really 
{'source': 'C:\\Users\\sacha\\jupyter\\RAG project\\Transcripts\\1.1 Lecture 1  What is a DB and Why Use It.txt'}


In [3]:
# Split into chunks 
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

split_docs = splitter.split_documents(raw_docs)

In [4]:
# Gemini embeddings 
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [5]:
# Create local vector Chroma DB
persist_dir = "./chroma_db"

if not os.path.exists(persist_dir):
    print("Building vector index...")
    
    vectordb = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        persist_directory=persist_dir,
        collection_name="test_rag",
    )
    print(f"Stored {len(split_docs)} chunks in {persist_dir}")
    
else:
    print("Loading existing vector index...")

    vectordb = Chroma(
        persist_directory=persist_dir,
        embedding_function=embeddings,
        collection_name="test_rag",
    )
    print(f"Loaded {len(split_docs)} chunks from {persist_dir}")

Loading existing vector index...
Loaded 80 chunks from ./chroma_db


## Retrieval

In [6]:
# Retriever creation, k is the number of retrieved chunks 
retriever = vectordb.as_retriever(
    search_kwargs={"k": 3}
)

In [7]:
# test 
query = "Explain the history of databases"

docs = retriever.invoke(query)

for i, d in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", d.metadata.get("source"))
    print(d.page_content[:300])


--- Result 1 ---
Source: C:\Users\sacha\jupyter\RAG project\Transcripts\1.1 Lecture 2  Mini-history.txt
Before we get into the technical content of the course I want to start with a little bit of history. Why Well first of all I just like it it's an interesting history. And secondly because it's going to provide us with a bit of context for some of the decisions of how databases are structured. We're 

--- Result 2 ---
Source: C:\Users\sacha\jupyter\RAG project\Transcripts\1.1 Lecture 2  Mini-history.txt
to turn this logical description into some executable code. This is a descriptive notion of what we want to do. So moving a little later into more modern history. Well there is some other notions in particular object oriented databases. Write object oriented languages were getting really popular dur

--- Result 3 ---
Source: C:\Users\sacha\jupyter\RAG project\Transcripts\1.1 Lecture 2  Mini-history.txt
at this point. Now this idea though has returned with more complex objects and with

## Generation

In [8]:
# Pick LLM 
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.0-flash-001", 
    temperature=0.2,
)

In [20]:
def format_docs(docs):
    """
    Function merging docs in a string 
    """
    return "\n\n---\n\n".join(d.page_content for d in docs)

def source(query):
    """
    Function getting sources for a query
    """
    docs = retriever.invoke(query)
    seen = set()
    print(f"\n--- Source ---")
    for d in docs:
        source = os.path.basename(d.metadata.get("source"))
        if source and source not in seen:
            seen.add(source)
            print(source[:-4])

In [10]:
# Template 
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer the user's question using ONLY the provided context. "
     "If the answer is not in the context, say: 'I don't know based on the provided context.'"),
    ("human",
     "Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:")
])

In [11]:
# Create a dictionary based on the vector db and feed it to the LLM with the prompt template 
rag_chain = (
    {
        "context": retriever | format_docs,   
        "question": RunnablePassthrough(), 
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [21]:
question = "Explain what is a key, what is a super key, and how they are different than each other."

answer = rag_chain.invoke(question)

print(answer)
source(question)

A super key is a set of attributes such that if the tuples have the same super key values, then the rest of the tuples are the same as well. A key, also known as a candidate key, is a minimal super key, meaning no subset of it is also a key. Keys are subsets of super keys, specifically the minimal subsets. If you have duplicate tuples, your table does not have a super key or a key.

--- Source ---
1.1 Lecture 7  Keys & primary keys part 2
